## 1) Initialization and Config

Imports, project path setup, global configuration, and runtime constants.

In [12]:
from __future__ import annotations

%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import gc
import logging
import os
import random
import sys
from pathlib import Path
from typing import Any

import numpy as np
import torch
import tqdm
import wandb
from dotenv import load_dotenv
from torch.utils.data import DataLoader
from torchinfo import summary

# Resolve project root robustly for notebook execution from either repo root or this folder.
if (Path.cwd() / "vae_features").exists():
    PROJECT_ROOT = Path.cwd()
elif Path.cwd().name == "train" and (Path.cwd().parent / "utils").exists():
    PROJECT_ROOT = Path.cwd().parents[1]
else:
    PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from vae_features.loss.hyperSphericalLoss import HypersphericalVAELoss
from vae_features.model.feedForwardVae import FeedForwardVAE
from vae_features.model.graphVae import GraphVAE
from vae_features.train.latent_visualization import LatentProjectionImageList, LatentProjectionVisualizer
from vae_features.train.mhr_pose_dataset import MHRPoseDataset, MHRBatch, pose_image_abs_path
from vae_features.train.reconstruction_visualization import ReconstructionImageList, ReconstructionVisualizer
from vae_features.utils.angleFormat import rotation_6d_to_euler, wrap_euler_angles_pi
from vae_features.utils.feedForward import Norm
from vae_features.utils.skeletonFormat import SkeletonFormat

CONFIG = {
    "NUM_EPOCHS": 20,
    "BATCH_SIZE": 128,
    "LEARNING_RATE": 7e-4,
    "WEIGHT_DECAY": 1e-7,
    "USE_6D_ROTATIONS": False,
    # If True (Euler-only): angle loss uses sin/cos matching instead of MSE on angles.
    "USE_TRIG_EULER_ANGLE_LOSS": True,
    "USE_GRAPH_VAE": False,
    "USE_VERTEX_SUPERVISION": True,

    # LR schedule: linear warmup, then cosine annealing (period = COSINE_T_MAX_EPOCHS per PyTorch CosineAnnealingLR)
    "WARMUP_EPOCHS": 3,
    "COSINE_T_MAX_EPOCHS": 5,
    "COSINE_ETA_MIN": 1e-6,

    # To avoid gradient explosion. Set to 1 to disable
    "GRAD_ACCUMULATION_STEPS": 1,
    # Clip gradients to avoid explosion
    "CLIP_GRADIENTS": False,
    "EMPTY_CACHE_AFTER_BATCH": True,

    "KL_WEIGHT": 5e-6,
    "NUM_EPOCHS_TO_FULL_KL": 5,
    "VERTEX_LOSS_WEIGHT": 5e-8,
    "NUM_EPOCHS_TO_FULL_VERTEX": 5,

    "DATA_DIR": str(PROJECT_ROOT / "data"),
    "TRAIN_RATIO": 0.99,
    "TOTAL_DATAPOINTS": None,

    "USE_WANDB": True,
    "WANDB_RUN_NAME": "giga_FF_New_after_graph_2056_128",
    "WANDB_PREVIOUS_RUN_ID": None,
    "WANDB_PROJECT_NAME": "multimodal_2025",

    # Validation artifacts
    "PCA_THUMBNAIL_ZOOM": 0.8,
    "CLEAR_VALIDATION_IMAGES_ON_START": True,

    "INITIAL_CONCENTRATION": 50.0,

    # Model hyperparameters
    "GRAPH_NUM_LAYERS": 2,
    "GRAPH_JOINT_EMBED_DIM": 64,
    "GRAPH_BONE_EMBED_DIM": 64,
    "GRAPH_DECODER_JOINT_EMBED_DIM": 64,
    "GRAPH_NUM_HEADS": 8,
    "GRAPH_BOTTLENECK_DIM": 128,
    "GRAPH_DROPOUT": 0.1,

    "FF_ENCODER_SIZES": [2056, 1024, 512, 256],
    "FF_DROPOUT": 0.1,
    "FF_NORMALIZATION": "layer",
    "FF_USE_RESIDUALS": False,

    "CHECKPOINT_BASENAME": "mhr_vae",
    "SEED": 42,
}

TRAIN_DIR = PROJECT_ROOT / "vae_features" / "train"
SKELETON_JSON_PATH = TRAIN_DIR / "mhr_skeleton_format.json"
JOINT_NAMES_JSON_PATH = TRAIN_DIR / "joint_names.json"
PARQUET_PATH = Path(CONFIG["DATA_DIR"]) / "processed_poses.parquet"
MHR_MODEL_PT_PATH = PROJECT_ROOT / "checkpoints" / "sam3d" / "dinov3" / "assets" / "mhr_model.pt"
VALIDATION_DIR = TRAIN_DIR / "validation"
PCA_DIR = VALIDATION_DIR / "PCA"
RECONSTRUCTION_DIR = VALIDATION_DIR / "reconstruction"
CHECKPOINT_DIR = TRAIN_DIR / "checkpoints"

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("train_vae")
logger.info(f"Using device: {DEVICE}")

INFO:train_vae:Using device: cuda


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2) Reproducibility and Output Paths

Set random seeds, create output directories, optionally clear prior PCA/reconstruction images (`CLEAR_VALIDATION_IMAGES_ON_START` in `CONFIG`), and validate data availability.

In [13]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_output_dirs():
    for path in [PCA_DIR, RECONSTRUCTION_DIR, CHECKPOINT_DIR]:
        path.mkdir(parents=True, exist_ok=True)


def clear_validation_image_artifacts() -> None:
    """Delete image files under PCA and reconstruction dirs (png/jpg/webp/gif)."""
    exts = {".png", ".jpg", ".jpeg", ".webp", ".gif"}
    removed = 0
    for d in (PCA_DIR, RECONSTRUCTION_DIR):
        if not d.is_dir():
            continue
        for f in d.iterdir():
            if f.is_file() and f.suffix.lower() in exts:
                f.unlink()
                removed += 1
    if removed:
        logger.info("Cleared %s prior validation image(s) from PCA/reconstruction", removed)
    else:
        logger.info("No prior validation images found under PCA/reconstruction to clear")


set_seed(CONFIG["SEED"])
make_output_dirs()
if CONFIG["CLEAR_VALIDATION_IMAGES_ON_START"]:
    clear_validation_image_artifacts()

if not PARQUET_PATH.exists():
    raise FileNotFoundError(f"Missing parquet file: {PARQUET_PATH}")

logger.info(f"Parquet file: {PARQUET_PATH}")

INFO:train_vae:Cleared 40 prior validation image(s) from PCA/reconstruction
INFO:train_vae:Parquet file: /home/ness/Nexus/school/10315-Art-ML/data/processed_poses.parquet


## 3) Dataset and DataLoaders

Load skeleton format + parquet dataset and create train/validation data loaders.

In [14]:
skeleton_format = SkeletonFormat.from_json_file(SKELETON_JSON_PATH)

dataset = MHRPoseDataset(
    parquet_path=PARQUET_PATH,
    skeleton_format=skeleton_format,
    joint_names_path=JOINT_NAMES_JSON_PATH,
    data_root=CONFIG["DATA_DIR"],
    max_samples=CONFIG["TOTAL_DATAPOINTS"],
    device=torch.device(DEVICE),
    use_6d_rotations=CONFIG["USE_6D_ROTATIONS"],
)

total_samples = len(dataset)
num_train = int(total_samples * CONFIG["TRAIN_RATIO"])
num_val = total_samples - num_train

if num_train <= 0 or num_val <= 0:
    raise ValueError(
        f"Invalid split from {total_samples} samples with TRAIN_RATIO={CONFIG['TRAIN_RATIO']}"
    )

generator = torch.Generator().manual_seed(CONFIG["SEED"])
train_dataset, val_dataset = torch.utils.data.random_split(
    dataset,
    [num_train, num_val],
    generator=generator,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["BATCH_SIZE"],
    shuffle=True,
    num_workers=0,
    pin_memory=DEVICE == "cuda",
    collate_fn=dataset.collate_fn,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["BATCH_SIZE"],
    shuffle=False,
    num_workers=0,
    pin_memory=DEVICE == "cuda",
    collate_fn=dataset.collate_fn,
)

logger.info(f"Train samples: {len(train_dataset)}")
logger.info(f"Val samples: {len(val_dataset)}")

INFO:train_vae:Train samples: 29328
INFO:train_vae:Val samples: 297


## 4) Model, Loss, Optimizer, and Scheduler

Initialize GraphVAE/FeedForwardVAE, hyperspherical loss, optimizer, and LR schedule: linear warmup then cosine annealing.

In [15]:
if CONFIG["USE_GRAPH_VAE"]:
    model = GraphVAE(
        skeleton_format=skeleton_format,
        num_layers=CONFIG["GRAPH_NUM_LAYERS"],
        joint_embedding_dimension=CONFIG["GRAPH_JOINT_EMBED_DIM"],
        bone_embedding_dimension=CONFIG["GRAPH_BONE_EMBED_DIM"],
        decoder_joint_embedding_dimension=CONFIG["GRAPH_DECODER_JOINT_EMBED_DIM"],
        num_attention_heads=CONFIG["GRAPH_NUM_HEADS"],
        bottleneck_dimensions=CONFIG["GRAPH_BOTTLENECK_DIM"],
        bottleneck_activation=torch.nn.GELU(),
        dropout=CONFIG["GRAPH_DROPOUT"],
        use_6d_rotation_format=CONFIG["USE_6D_ROTATIONS"],
        initial_concentration=CONFIG["INITIAL_CONCENTRATION"],
        device=DEVICE
    )
else:
    norm: Norm = 'layerNorm' if CONFIG["FF_NORMALIZATION"].lower() == "layer" else 'batchNorm'
    model = FeedForwardVAE(
        skeletonFormat=skeleton_format,
        encoderSizes=CONFIG["FF_ENCODER_SIZES"],
        dropout=CONFIG["FF_DROPOUT"],
        use_residuals=CONFIG["FF_USE_RESIDUALS"],
        activation=torch.nn.GELU(),
        normalization=norm,
        device=DEVICE,
        use_6d_rotation_format=CONFIG["USE_6D_ROTATIONS"],
        initial_concentration=CONFIG["INITIAL_CONCENTRATION"]
    )

model = model.to(DEVICE).float()
logger.info(f"Model initialized: {type(model).__name__}")

INFO:train_vae:Model initialized: FeedForwardVAE


In [16]:
MODEL_PARAM_COUNT = sum(p.numel() for p in model.parameters())
logger.info(f"Model parameter count: {MODEL_PARAM_COUNT:,}")

class _ModelSummaryWrapper(torch.nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, x):
        x = x.to(DEVICE)
        if CONFIG["USE_GRAPH_VAE"]:
            _, latent, _ = self.base_model.encode(x)
            return self.base_model.decode(latent)
        flat = x.reshape(x.shape[0], -1)
        _, latent, _ = self.base_model.encode(flat)
        return self.base_model.decode(latent)


summary_wrapper = _ModelSummaryWrapper(model)
if CONFIG["USE_GRAPH_VAE"]:
    summary_input_shape = (1, skeleton_format.get_joint_count(), 6 if CONFIG["USE_6D_ROTATIONS"] else 3)
else:
    summary_input_shape = (1, skeleton_format.get_joint_count() * (6 if CONFIG["USE_6D_ROTATIONS"] else 3))
summary(summary_wrapper, input_size=summary_input_shape, device=DEVICE)

INFO:train_vae:Model parameter count: 7,111,297


Layer (type:depth-idx)                   Output Shape              Param #
_ModelSummaryWrapper                     [1, 381]                  --
├─FeedForwardVAE: 1-1                    --                        --
│    └─FeedForward: 2-9                  --                        (recursive)
│    │    └─Sequential: 3-1              [1, 257]                  3,556,099
│    └─FeedForward: 2-10                 --                        (recursive)
│    │    └─Sequential: 3-8              --                        (recursive)
│    └─FeedForward: 2-9                  --                        (recursive)
│    │    └─Sequential: 3-9              --                        (recursive)
│    └─FeedForward: 2-10                 --                        (recursive)
│    │    └─Sequential: 3-8              --                        (recursive)
│    └─FeedForward: 2-9                  --                        (recursive)
│    │    └─Sequential: 3-9              --                        (recursiv

In [17]:
criterion = HypersphericalVAELoss(
    use_6d_rotation_format=CONFIG["USE_6D_ROTATIONS"],
    device=DEVICE,
    use_vertex_supervision=CONFIG["USE_VERTEX_SUPERVISION"],
    mhr_model_path=str(MHR_MODEL_PT_PATH) if CONFIG["USE_VERTEX_SUPERVISION"] else None,
    vertex_loss_weight=float(CONFIG["VERTEX_LOSS_WEIGHT"]),
    use_trig_euler_angle_loss=CONFIG["USE_TRIG_EULER_ANGLE_LOSS"],
).to(DEVICE)

In [18]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["LEARNING_RATE"],
    weight_decay=CONFIG["WEIGHT_DECAY"],
)

warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=0.01,
    end_factor=1.0,
    total_iters=CONFIG["WARMUP_EPOCHS"],
)
_cosine_t_max = max(1, int(CONFIG["COSINE_T_MAX_EPOCHS"]))
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=_cosine_t_max,
    eta_min=float(CONFIG["COSINE_ETA_MIN"]),
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[CONFIG["WARMUP_EPOCHS"]],
)


def kl_schedule(epoch: int) -> float:
    epoch = epoch + 1

    if epoch < CONFIG["NUM_EPOCHS_TO_FULL_KL"]:
        return float(
            CONFIG["KL_WEIGHT"]) * (epoch / CONFIG["NUM_EPOCHS_TO_FULL_KL"])

    return float(CONFIG["KL_WEIGHT"])


def vertex_loss_schedule(epoch: int) -> float:
    """Warm up vertex loss weight from 0 to VERTEX_LOSS_WEIGHT (same pattern as KL)."""
    if not CONFIG["USE_VERTEX_SUPERVISION"]:
        return 0.0
    epoch = epoch + 1
    if epoch < CONFIG["NUM_EPOCHS_TO_FULL_VERTEX"]:
        return float(
            CONFIG["VERTEX_LOSS_WEIGHT"]
            * (epoch / CONFIG["NUM_EPOCHS_TO_FULL_VERTEX"])
        )
    return float(CONFIG["VERTEX_LOSS_WEIGHT"])


## 5) Training Loop

Define forward/reconstruction behavior and train one epoch with KL scheduling and gradient controls.

In [19]:
def vae_output_to_mhr_params(
    reconstructed_joint_output: torch.Tensor,
    original_raw: torch.Tensor,
    inverse_reorder_indices: torch.Tensor,
    post_processor,
    use_6d_rotations: bool,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Differentiable helper that maps VAE outputs back to raw and MHR parameters.

    Flat (B, J*D) tensors are reshaped using D=6 when use_6d_rotations else D=3.
    When use_6d_rotations, 6D rotations are converted to Euler before MHR fitting.
    """
    if reconstructed_joint_output.ndim == 2:
        rot_dim = 6 if use_6d_rotations else 3
        reconstructed_joint_output = reconstructed_joint_output.view(
            reconstructed_joint_output.shape[0],
            inverse_reorder_indices.shape[0],
            rot_dim,
        )

    if reconstructed_joint_output.ndim != 3:
        raise ValueError(
            f"Expected reconstructed output with shape (B, J, D), got {reconstructed_joint_output.shape}"
        )

    if use_6d_rotations:
        if reconstructed_joint_output.shape[-1] != 6:
            raise ValueError(
                f"Expected 6D reconstructed rotations when use_6d_rotations=True, got {reconstructed_joint_output.shape}"
            )
        euler_joint_angles = rotation_6d_to_euler(reconstructed_joint_output)
    else:
        if reconstructed_joint_output.shape[-1] != 3:
            raise ValueError(
                f"Expected Euler reconstructed rotations with last dim 3, got {reconstructed_joint_output.shape}"
            )
        euler_joint_angles = reconstructed_joint_output

    inverse_idx = inverse_reorder_indices.to(euler_joint_angles.device)
    euler_joint_angles_mhr_order = euler_joint_angles[:, inverse_idx, :]

    patched_raw, reconstructed_mhr_params = post_processor.joint_angles_to_mhr_parameters(
        joint_angles_mhr_order=euler_joint_angles_mhr_order,
        base_raw=original_raw,
    )
    return patched_raw, reconstructed_mhr_params

In [20]:
def forward_and_reconstruct(batch_angles: torch.Tensor):
    if CONFIG["USE_GRAPH_VAE"]:
        distribution, mean_latent, z, concentration, reconstruction = model.encode_and_reconstruct(batch_angles)
        target = batch_angles
        return distribution, z, reconstruction, target

    flat_input = batch_angles.reshape(batch_angles.shape[0], -1)
    distribution, mean_latent, z, concentration, reconstruction = model.encode_and_reconstruct(flat_input)
    target = flat_input
    return distribution, z, reconstruction, target


def euler_reconstruction_and_target(
    reconstruction: torch.Tensor,
    target: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Before trig loss / MHR: wrap raw Euler outputs to (-pi, pi] (differentiable)."""
    if CONFIG["USE_6D_ROTATIONS"]:
        return reconstruction, target
    return wrap_euler_angles_pi(reconstruction), wrap_euler_angles_pi(target)


def train_epoch(epoch: int) -> dict[str, float]:
    model.train()
    progress = tqdm.tqdm(train_loader, desc=f"Train Epoch {epoch + 1}")

    total_loss = 0.0
    total_angle_rec_loss = 0.0
    total_kl_loss = 0.0
    total_vertex_loss = 0.0
    batches_seen = 0

    optimizer.zero_grad()
    for batch_idx, batch in enumerate(progress):
        batch = batch
        joint_angles = batch.joint_angles.to(DEVICE)

        if torch.isnan(joint_angles).any():
            logger.warning(f"NaN in train batch {batch_idx}, skipping")
            continue

        latent_distribution, _, reconstruction, target = forward_and_reconstruct(
            joint_angles)
        reconstruction, target = euler_reconstruction_and_target(reconstruction, target)

        reconstructed_mhr_params = None
        target_mhr_params = None
        if CONFIG["USE_VERTEX_SUPERVISION"]:
            _, reconstructed_mhr_params = vae_output_to_mhr_params(
                reconstructed_joint_output=reconstruction,
                original_raw=batch.joint_data.raw,
                inverse_reorder_indices=dataset.inverse_reorder_indices,
                post_processor=dataset.post_processor,
                use_6d_rotations=CONFIG["USE_6D_ROTATIONS"],
            )
            target_mhr_params = torch.stack(
                [meta.mhr_parameters for meta in batch.metadata],
                dim=0).to(DEVICE)

        loss_dict = criterion(
            predicted_joint_angles=reconstruction,
            label_joint_angles=target,
            latent_distributions=latent_distribution,
            kl_weight=kl_schedule(epoch),
            reconstructed_mhr_params=reconstructed_mhr_params,
            target_mhr_params=target_mhr_params,
            vertex_loss_weight=vertex_loss_schedule(epoch),
        )
        loss = loss_dict["total_loss"]

        if torch.isnan(loss):
            logger.warning(f"NaN loss in train batch {batch_idx}, skipping")
            optimizer.zero_grad()
            continue

        scaled_loss = loss / CONFIG["GRAD_ACCUMULATION_STEPS"]
        scaled_loss.backward()

        if (batch_idx + 1) % CONFIG["GRAD_ACCUMULATION_STEPS"] == 0:
            if CONFIG["CLIP_GRADIENTS"]:
                torch.nn.utils.clip_grad_norm_(model.parameters(),
                                               max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        if CONFIG["EMPTY_CACHE_AFTER_BATCH"] and DEVICE == "cuda":
            torch.cuda.empty_cache()
            gc.collect()
        elif CONFIG["EMPTY_CACHE_AFTER_BATCH"] and DEVICE == "mps":
            torch.mps.empty_cache()
            gc.collect()

        total_loss += float(loss.item())
        total_angle_rec_loss += float(loss_dict["angle_rec_loss"].item())
        total_kl_loss += float(loss_dict["kl_loss"].item())
        total_vertex_loss += float(loss_dict["vertex_loss"].item())
        batches_seen += 1
        progress.set_postfix({
            "total_loss":
            total_loss / max(batches_seen, 1),
            "angle_rec_loss":
            total_angle_rec_loss / max(batches_seen, 1),
            "kl_loss":
            total_kl_loss / max(batches_seen, 1),
            "vertex_loss":
            total_vertex_loss / max(batches_seen, 1),
        })

    if len(train_loader) % CONFIG["GRAD_ACCUMULATION_STEPS"] != 0:
        if CONFIG["CLIP_GRADIENTS"]:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        optimizer.zero_grad()

    avg_loss = total_loss / max(batches_seen, 1)
    avg_angle_rec_loss = total_angle_rec_loss / max(batches_seen, 1)
    avg_kl_loss = total_kl_loss / max(batches_seen, 1)
    avg_vertex_loss = total_vertex_loss / max(batches_seen, 1)
    return {
        "train_loss": avg_loss,
        "train_angle_rec_loss": avg_angle_rec_loss,
        "train_kl_loss": avg_kl_loss,
        "train_vertex_loss": avg_vertex_loss,
        "kl_weight": kl_schedule(epoch),
        "vertex_loss_weight": vertex_loss_schedule(epoch),
        "learning_rate": scheduler.get_last_lr()[0],
    }

## 6) Validation: Latent Projection + Reconstruction Visualization

After the val-loader loss pass, load **JSON-listed image paths** (`latent_projection_images.json`, `reconstruction_images.json`), resolve rows from the **full parquet** via `MHRPoseDataset.get_batch_by_image_paths` (not only the val split), run the model on those batches, then save **UMAP (cosine)** latent thumbnails and reconstruction triptychs.

In [21]:
def validate_epoch(epoch: int) -> dict[str, Any]:
    model.eval()

    total_loss = 0.0
    total_angle_rec_loss = 0.0
    total_kl_loss = 0.0
    total_vertex_loss = 0.0
    batches_seen = 0

    with torch.no_grad():
        progress = tqdm.tqdm(val_loader, desc=f"Val Epoch {epoch + 1}")
        for batch_idx, batch in enumerate(progress):
            batch = batch  # type: MHRBatch
            joint_angles = batch.joint_angles.to(DEVICE)

            if torch.isnan(joint_angles).any():
                logger.warning(f"NaN in val batch {batch_idx}, skipping")
                continue

            latent_distribution, latent, reconstruction, target = forward_and_reconstruct(joint_angles)
            reconstruction, target = euler_reconstruction_and_target(reconstruction, target)

            reconstructed_mhr_params = None
            target_mhr_params = None
            if CONFIG["USE_VERTEX_SUPERVISION"]:
                _, reconstructed_mhr_params = vae_output_to_mhr_params(
                    reconstructed_joint_output=reconstruction,
                    original_raw=batch.joint_data.raw,
                    inverse_reorder_indices=dataset.inverse_reorder_indices,
                    post_processor=dataset.post_processor,
                    use_6d_rotations=CONFIG["USE_6D_ROTATIONS"],
                )
                target_mhr_params = torch.stack(
                    [meta.mhr_parameters for meta in batch.metadata], dim=0
                ).to(DEVICE)

            loss_dict = criterion(
                predicted_joint_angles=reconstruction,
                label_joint_angles=target,
                latent_distributions=latent_distribution,
                kl_weight=kl_schedule(epoch),
                reconstructed_mhr_params=reconstructed_mhr_params,
                target_mhr_params=target_mhr_params,
                vertex_loss_weight=vertex_loss_schedule(epoch),
            )
            loss = loss_dict["total_loss"]
            if torch.isnan(loss):
                continue

            total_loss += float(loss.item())
            total_angle_rec_loss += float(loss_dict["angle_rec_loss"].item())
            total_kl_loss += float(loss_dict["kl_loss"].item())
            total_vertex_loss += float(loss_dict["vertex_loss"].item())
            batches_seen += 1
            progress.set_postfix(
                {
                    "total_loss": total_loss / max(batches_seen, 1),
                    "angle_rec_loss": total_angle_rec_loss / max(batches_seen, 1),
                    "kl_loss": total_kl_loss / max(batches_seen, 1),
                    "vertex_loss": total_vertex_loss / max(batches_seen, 1),
                }
            )

            if CONFIG["EMPTY_CACHE_AFTER_BATCH"] and DEVICE == "cuda":
                torch.cuda.empty_cache()
                gc.collect()

    data_root = Path(CONFIG["DATA_DIR"])

    # Latent UMAP (cosine) / thumbnails: JSON paths resolved against full parquet (not val-split only)
    pca_path = None
    latent_paths = LatentProjectionImageList().get_image_paths(CONFIG["DATA_DIR"])
    vis_latent_batch, latent_missing = dataset.get_batch_by_image_paths(latent_paths)
    if latent_missing:
        logger.warning(
            "Latent projection: %d image paths not in parquet (showing up to 5): %s",
            len(latent_missing),
            latent_missing[:5],
        )
    if vis_latent_batch is not None and len(vis_latent_batch.metadata) >= 2:
        ja = vis_latent_batch.joint_angles.to(DEVICE)
        _, latent_vis, _, _ = forward_and_reconstruct(ja)
        emb = latent_vis.detach().cpu().numpy()
        emb_rows: list[np.ndarray] = []
        thumb_paths: list[str] = []
        for row, meta in zip(emb, vis_latent_batch.metadata, strict=False):
            pth = pose_image_abs_path(meta, data_root)
            if pth is None:
                continue
            emb_rows.append(row)
            thumb_paths.append(pth)
        if len(thumb_paths) >= 2:
            emb_f = np.stack(emb_rows, axis=0)
            pca_path = LatentProjectionVisualizer().save_projection(
                embeddings=emb_f,
                image_paths=thumb_paths,
                output_path=PCA_DIR / f"epoch_{epoch + 1}_latent_projection.png",
                title=f"Latent UMAP (cosine, epoch {epoch + 1})",
                method="umap",
                thumbnail_zoom=float(CONFIG["PCA_THUMBNAIL_ZOOM"]),
            )
        else:
            logger.warning("Latent projection skipped: fewer than 2 images on disk for JSON paths")
    else:
        logger.warning("Latent projection skipped: need at least 2 parquet rows for JSON paths")

    # Reconstruction triptychs: same pattern
    reconstruction_paths: list[str] = []
    recon_paths = ReconstructionImageList().get_image_paths(CONFIG["DATA_DIR"])
    recon_batch, recon_missing = dataset.get_batch_by_image_paths(recon_paths)
    if recon_missing:
        logger.warning(
            "Reconstruction: %d image paths not in parquet (showing up to 5): %s",
            len(recon_missing),
            recon_missing[:5],
        )
    if recon_batch is not None:
        ja = recon_batch.joint_angles.to(DEVICE)
        _, _, reconstruction, _ = forward_and_reconstruct(ja)
        reconstruction, _ = euler_reconstruction_and_target(reconstruction, reconstruction)
        _, reconstructed_mhr_params = vae_output_to_mhr_params(
            reconstructed_joint_output=reconstruction,
            original_raw=recon_batch.joint_data.raw,
            inverse_reorder_indices=dataset.inverse_reorder_indices,
            post_processor=dataset.post_processor,
            use_6d_rotations=CONFIG["USE_6D_ROTATIONS"],
        )
        recon_samples: list[dict[str, Any]] = []
        for meta, reconstructed_mhr in zip(recon_batch.metadata, reconstructed_mhr_params, strict=False):
            recon_samples.append(
                {
                    "image_path": meta.image_path,
                    "image_path_abs": meta.image_path_abs,
                    "pred_cam": meta.pred_cam,
                    "pred_cam_t": meta.pred_cam_t,
                    "focal_length": meta.focal_length,
                    "pred_keypoints_2d": meta.pred_keypoints_2d,
                    "original_mhr_parameters": meta.mhr_parameters,
                    "reconstructed_mhr_parameters": reconstructed_mhr,
                }
            )
        reconstruction_paths = ReconstructionVisualizer().save(
            samples=recon_samples,
            epoch=epoch,
            output_dir=RECONSTRUCTION_DIR,
            project_root=PROJECT_ROOT,
            data_dir=CONFIG["DATA_DIR"],
        )

    return {
        "val_loss": total_loss / max(batches_seen, 1),
        "val_angle_rec_loss": total_angle_rec_loss / max(batches_seen, 1),
        "val_kl_loss": total_kl_loss / max(batches_seen, 1),
        "val_vertex_loss": total_vertex_loss / max(batches_seen, 1),
        "pca_path": pca_path,
        "reconstruction_paths": reconstruction_paths,
    }


## 7) WandB and Full Training Run

Initialize WandB, run epoch training/validation, log artifacts, and save checkpoints.

In [22]:
load_dotenv()
WANDB_API_KEY = os.environ.get("WANDB_API_KEY")

run = None
if CONFIG["USE_WANDB"]:
    if WANDB_API_KEY:
        wandb.login(key=WANDB_API_KEY)
    else:
        logger.warning("WANDB_API_KEY missing; wandb may prompt for login")

    resume_logging = CONFIG["WANDB_PREVIOUS_RUN_ID"] is not None
    if resume_logging:
        run = wandb.init(
            settings=wandb.Settings(symlink=False),
            id=CONFIG["WANDB_PREVIOUS_RUN_ID"],
            resume="must",
            project=CONFIG["WANDB_PROJECT_NAME"],
            config=CONFIG,
        )
    else:
        run = wandb.init(
            name=CONFIG["WANDB_RUN_NAME"],
            reinit=True,
            project=CONFIG["WANDB_PROJECT_NAME"],
            config=CONFIG,
        )

    wandb.log({"model_param_count": MODEL_PARAM_COUNT})

best_val_loss = float("inf")
history: list[dict[str, float]] = []

for epoch in range(CONFIG["NUM_EPOCHS"]):
    logger.info(
        f"Epoch {epoch + 1}/{CONFIG['NUM_EPOCHS']} "
        f"kl_weight={kl_schedule(epoch):.6f} vertex_loss_weight={vertex_loss_schedule(epoch):.6f}"
    )
    train_metrics = train_epoch(epoch)
    val_report = validate_epoch(epoch)

    scheduler.step()

    epoch_metrics = {
        **train_metrics,
        "val_loss": val_report["val_loss"],
        "val_angle_rec_loss": val_report["val_angle_rec_loss"],
        "val_kl_loss": val_report["val_kl_loss"],
        "val_vertex_loss": val_report["val_vertex_loss"],
        "epoch": epoch + 1,
    }
    history.append(epoch_metrics)

    logger.info(
        f"train_loss={epoch_metrics['train_loss']:.6f} val_loss={epoch_metrics['val_loss']:.6f} "
        f"lr={scheduler.get_last_lr()[0]:.6f}"
    )

    latest_ckpt = CHECKPOINT_DIR / f"{CONFIG['CHECKPOINT_BASENAME']}_latest.pt"
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "config": CONFIG,
            "history": history,
            "epoch": epoch + 1,
        },
        latest_ckpt,
    )

    if epoch_metrics["val_loss"] < best_val_loss:
        best_val_loss = epoch_metrics["val_loss"]
        best_ckpt = CHECKPOINT_DIR / f"{CONFIG['CHECKPOINT_BASENAME']}_best.pt"
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "config": CONFIG,
                "history": history,
                "epoch": epoch + 1,
            },
            best_ckpt,
        )
        logger.info(f"Saved best checkpoint: {best_ckpt}")

    if run is not None:
        log_payload: dict[str, Any] = {
            "train_loss": epoch_metrics["train_loss"],
            "train_angle_rec_loss": epoch_metrics["train_angle_rec_loss"],
            "train_kl_loss": epoch_metrics["train_kl_loss"],
            "train_vertex_loss": epoch_metrics["train_vertex_loss"],
            "val_loss": epoch_metrics["val_loss"],
            "val_angle_rec_loss": epoch_metrics["val_angle_rec_loss"],
            "val_kl_loss": epoch_metrics["val_kl_loss"],
            "val_vertex_loss": epoch_metrics["val_vertex_loss"],
            "kl_weight": epoch_metrics["kl_weight"],
            "vertex_loss_weight": epoch_metrics["vertex_loss_weight"],
            "learning_rate": scheduler.get_last_lr()[0],
            "epoch": epoch + 1,
        }

        if val_report["pca_path"] is not None and Path(val_report["pca_path"]).exists():
            log_payload["latent_projection"] = wandb.Image(val_report["pca_path"])

        for i, recon_path in enumerate(val_report.get("reconstruction_paths", [])[:3]):
            if Path(recon_path).exists():
                log_payload[f"reconstruction_{i}"] = wandb.Image(recon_path)

        wandb.log(log_payload)

if run is not None:
    run.finish()

logger.info("Training complete.")
logger.info(f"Checkpoints saved under: {CHECKPOINT_DIR}")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/ness/.netrc


INFO:train_vae:Epoch 1/20 kl_weight=0.000001 vertex_loss_weight=0.000000
Val Epoch 1: 100%|██████████| 3/3 [00:01<00:00,  2.44it/s, total_loss=3.66, angle_rec_loss=3.35, kl_loss=522, vertex_loss=3.09e+7]
/home/ness/Nexus/school/10315-Art-ML/.venv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
INFO:train_vae:train_loss=13.593104 val_loss=3.662952 lr=0.000238
INFO:train_vae:Saved best checkpoint: /home/ness/Nexus/school/10315-Art-ML/vae_features/train/checkpoints/mhr_vae_best.pt
INFO:train_vae:Epoch 2/20 kl_weight=0.000002 vertex_loss_weight=0.000000
Train Epoch 2:   9%|▊         | 20/230 [00:05<00:59,  3.52it/s, total_loss=7.04, angle_rec_loss=6.29, kl_loss=463, vertex_loss=3.78e+7]


ValueError: Expected parameter concentration (Tensor of shape (128, 2)) of distribution Dirichlet(concentration: torch.Size([128, 2])) to satisfy the constraint IndependentConstraint(GreaterThan(lower_bound=0.0), 1), but found invalid values:
tensor([[     nan, 127.5000],
        [     nan, 127.5000],
        [     nan, 127.5000],
        [     nan, 127.5000],
        [     nan, 127.5000],
        [     nan, 127.5000],
        [     nan, 127.5000],
        [     nan, 127.5000],
        [     nan, 127.5000],
        [     nan, 127.5000],
        [     nan, 127.5000],
        [     nan, 127.5000],
        [     nan, 127.5000],
        [     nan, 127.5000],
        [     nan, 127.5000],
        [     nan, 127.5000],
        [     nan, 127.5000],
        [     nan, 127.5000],
        [     nan, 127.5000],
        [     nan, 127.5000],
        [     nan, 127.5000],
        [     nan, 127.5000],
        [     nan, 127.5000],
        [     nan, 127.5000],
        [     nan, 127.5000],
       